In [1]:
import pandas as pd

df = pd.read_parquet("../data/processed/complaints_2020-03_2021-03.parquet")

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df["product"].nunique())
print(df["complaint_what_happened"].str.slice(0, 300).head(3).tolist())

(35717, 5)
complaint_id               object
date_received              object
product                    object
sub_product                object
complaint_what_happened    object
dtype: object
complaint_id               0
date_received              0
product                    0
sub_product                0
complaint_what_happened    0
dtype: int64
9
['Despite my feedback from Complaint XXXX, Experian keeps harassing me about my dispute of personal information that should not be a fact in controversy. Once again, Experian has chosen to send me updates on what I can already presume to be the one giving rise to the aforementioned complaint, closed w', 'XXXX and Transunion should not be reporting that I am ( 30 ) days late on the LENDMARK partial account number XXXX. ( Please see page attached from my credit report. ) This account reports a {$0.00} balance. Please keep in mind that it would be impossible to be late on an account where nothing is ow', 'XXXX ; Experian and Transunion shou

In [2]:
text = df["complaint_what_happened"]
n = len(text)

pii_checks = {
    "xxxx_masks": text.str.contains(r"XX+", regex=True),
    "emails": text.str.contains(r"[\w.+-]+@[\w-]+\.[\w.]+", regex=True),
    "long_digits": text.str.contains(r"\d{8,}", regex=True),
    "phones": text.str.contains(r"\(?\d{3}\)?[\s.-]?\d{3}[\s.-]\d{4}", regex=True),
}

for name, mask in pii_checks.items():
    print(f"{name:12s} {mask.sum():7d}  ({mask.mean():.2%})")

print(df["sub_product"].value_counts(dropna=False).head(10))
print((df["sub_product"].str.strip() == "").sum())

xxxx_masks     28712  (80.39%)
emails             7  (0.02%)
long_digits        3  (0.01%)
phones             1  (0.00%)
sub_product
Credit reporting                              19582
General-purpose credit card or charge card     2494
Conventional home mortgage                     1740
Checking account                               1547
Other debt                                     1519
Credit card debt                               1203
I do not know                                  1089
Medical debt                                    848
Loan                                            670
Store credit card                               526
Name: count, dtype: int64
1


In [3]:
letters = text.str.count(r"[^\W\d_]")
latin = text.str.count(r"[A-Za-z]")
non_latin_ratio = 1 - latin / letters.where(letters > 0)

words = text.str.lower().str.findall(r"[a-z']+")
word_count = words.str.len()
COMMON = {"the", "and", "to", "i", "of", "a", "my", "is", "in", "that", "was", "for"}
common_ratio = words.apply(lambda w: sum(t in COMMON for t in w)) / word_count.where(word_count > 0)

flag_script = non_latin_ratio > 0.2
flag_words = (word_count >= 20) & (common_ratio < 0.05)
flagged = flag_script | flag_words

print("non-latin script:", flag_script.sum())
print("few common words:", flag_words.sum())
print("total flagged:", flagged.sum())
print(text[flagged].str.slice(0, 200).head(5).tolist())

non-latin script: 0
few common words: 242
total flagged: 242
['( XXXX ) XXXX < -- - PHONE numbers are NOT MINE -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- XXXX XXXX XXXX XX', 'XXXX XXXX XXXX XXXX XXXX, TX XXXX XXXX XXXX XXXX Account charged off. {$3100.00} written off. {$3100.00} past due as of XX/XX/2020. \n( XXXX ) XXXX Public Record XXXX XXXX XXXX XXXX XXXX XXXX Addresses', 'Army Airforce Exchange Credit Services Contact with fraudulent representative Request removal from ex-spouse account No access to account Suspect credit card Fraud.', 'all FCRA violations I dont not have all this : XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX Addresses XXXX XXXX XXXX XXXX XXXX, TX XXXX SINGLE FA', "This account was closed on XX/XX/XXXX XXXX XXXX  - XXXX XXXX  XXXX Last reported XX/XX/XXXX {$4900.00} Reported balance Credit limit {$4500.00} You're using

In [4]:
norm = text.str.strip()

dup_any = norm.duplicated(keep=False)
groups = df.assign(norm=norm)[dup_any].groupby("norm")["product"].nunique()

same_label = groups[groups == 1]
conflict = groups[groups > 1]

print("rows in duplicate groups:", dup_any.sum())
print("extra copies (would be removed):", norm.duplicated().sum())
print("groups, same product:", len(same_label))
print("groups, different products:", len(conflict))
print("rows in conflicting groups:", norm.isin(conflict.index).sum())

rows in duplicate groups: 5065
extra copies (would be removed): 3497
groups, same product: 1536
groups, different products: 32
rows in conflicting groups: 312


In [5]:
conflict_rows = norm.isin(conflict.index)
clean = df.assign(norm=norm)[~conflict_rows].drop_duplicates(subset="norm")

counts = clean["product"].value_counts()
dist = pd.DataFrame({"count": counts, "share": (counts / len(clean)).round(3)})

print("rows after dedup:", len(clean))
print(dist)
print("smallest class x5:", counts.min() * 5)

rows after dedup: 32188
                                                    count  share
product                                                         
Credit reporting, credit repair services, or ot...  16682  0.518
Debt collection                                      5027  0.156
Credit card or prepaid card                          3428  0.106
Mortgage                                             2701  0.084
Checking or savings account                          1893  0.059
Money transfer, virtual currency, or money service    878  0.027
Vehicle loan or lease                                 808  0.025
Payday loan, title loan, or personal loan             421  0.013
Student loan                                          350  0.011
smallest class x5: 1750


In [6]:
lengths = clean["norm"].str.split().str.len()

print(lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(0))
for L in [100, 200, 300, 500]:
    print(f"max_len={L}: {(lengths > L).mean():.1%} of complaints get cut")

count    32188.0
mean       194.0
std        207.0
min          2.0
50%        139.0
75%        248.0
90%        395.0
95%        496.0
99%        973.0
max       5613.0
Name: norm, dtype: float64
max_len=100: 65.3% of complaints get cut
max_len=200: 33.6% of complaints get cut
max_len=300: 17.8% of complaints get cut
max_len=500: 4.9% of complaints get cut


In [8]:
import pandas as pd

SPLITS_DIR = "../data/processed/splits"
parts = {n: pd.read_parquet(f"{SPLITS_DIR}/{n}.parquet") for n in ["train", "val", "test"]}

table = pd.DataFrame({n: p["product"].value_counts() for n, p in parts.items()}).fillna(0).astype(int)
table["total"] = table.sum(axis=1)
table = table.sort_values("total", ascending=False)

print(table)
print((table[["train", "val", "test"]] / table[["train", "val", "test"]].sum()).round(3))

                                                    train    val  test  total
product                                                                      
Credit reporting, credit repair services, or ot...  60531  12109  6633  79273
Debt collection                                     20366   4817  3094  28277
Credit card or prepaid card                         14383   2713  1520  18616
Mortgage                                            10734   2507  1590  14831
Checking or savings account                          8281   1904  1142  11327
Money transfer, virtual currency, or money service   4406   1067   878   6351
Vehicle loan or lease                                3223    668   360   4251
Payday loan, title loan, or personal loan            1937    369   201   2507
Student loan                                         1846    355   191   2392
                                                    train    val   test
product                                                               

In [9]:
train = pd.read_parquet("../data/processed/splits/train.parquet")
lengths = train["complaint_what_happened"].str.split().str.len()

print(lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(0))
print(f"cut at 400: {(lengths > 400).mean():.1%}")

count    125707.0
mean        202.0
std         253.0
min           1.0
50%         133.0
75%         243.0
90%         419.0
95%         586.0
99%        1212.0
max        5617.0
Name: complaint_what_happened, dtype: float64
cut at 400: 10.9%
